In [135]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import optuna

from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

In [99]:
# set seaborn style
sns.set_style("white")
sns.set_context("talk")

In [116]:
# set random seed
seed = 42

In [117]:
# import data
df = pd.read_csv("../data/processed/corpus_studio_processed.csv")
df.drop(
    columns=["performance_type", "group"], inplace=True
)  # for this study, only studio recordings were included
df.fillna(
    "", inplace=True
)  # replace NaN values for previous and next words with empty strings
df.head()

,word,previous_word,next_word,artist,song_type,song,aae_feature,aae_realization,time,social_group
0,hollered,conductor,,cannedheat,cover,bring_it_on_home,post-consonantal d,0,1960s,nonAA_US
1,found,,,jeffhealey,cover,blue_jean_blues,post-consonantal d,1,1980s,nonAA_nonUS
2,my,,old,jeffhealey,cover,blue_jean_blues,ai monophthongization,1,1980s,nonAA_nonUS
3,my,,,jeffhealey,cover,blue_jean_blues,ai monophthongization,1,1980s,nonAA_nonUS
4,there,,is,albertking,cover,as_the_years_go_passing_by,post-vocalic r,0,1960s,AA


In [118]:
# define target and predictor columns
target_col = "aae_realization"
X = df.loc[:, df.columns != target_col]
y = df.loc[:, target_col]

# define categorical features
cat_features = [
    "previous_word",
    "word",
    "next_word",
    "artist",
    "song_type",
    "song",
    "aae_feature",
    "time",
    "social_group",
]

In [119]:
# create train test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=seed
)

In [120]:
# hyperparameter tuning with optuna
def objective(trial):
    params = {
        "iterations": 1000,
        "learning_rate": trial.suggest_float("learning_rate", 1e-3, 0.1, log=True),
        "depth": trial.suggest_int("depth", 1, 10),
        "subsample": trial.suggest_float("subsample", 0.05, 1.0),
        "colsample_bylevel": trial.suggest_float("colsample_bylevel", 0.05, 1.0),
        "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 1, 100),
    }

    model = CatBoostClassifier(**params)

    model.fit(
        X_train,
        y_train,
        eval_set=[(X_test, y_test)],
        cat_features=cat_features,
        verbose=0,
        early_stopping_rounds=100,
    )

    preds = model.predict(X_test)
    pred_labels = np.rint(preds)
    score = f1_score(y_test, pred_labels)
    return score

In [121]:
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=50, timeout=600)
print("Number of completed trials: {}".format(len(study.trials)))
print("Best trial:")
trial = study.best_trial

print("\tBest Score: {}".format(trial.value))
print("\tBest Params: ")
for key, value in trial.params.items():
    print("    {}: {}".format(key, value))

[I 2024-02-22 18:28:46,599] A new study created in memory with name: no-name-67ba969d-2184-4559-84c7-afa128f4e2d5


[I 2024-02-22 18:28:50,799] Trial 0 finished with value: 0.9116867824551075 and parameters: {'learning_rate': 0.021049900043615746, 'depth': 2, 'subsample': 0.38261449814640053, 'colsample_bylevel': 0.1576450581002573, 'min_data_in_leaf': 58}. Best is trial 0 with value: 0.9116867824551075.
[I 2024-02-22 18:28:58,010] Trial 1 finished with value: 0.9254262416604893 and parameters: {'learning_rate': 0.020551224931018366, 'depth': 6, 'subsample': 0.09542713019536239, 'colsample_bylevel': 0.22419638791403973, 'min_data_in_leaf': 94}. Best is trial 1 with value: 0.9254262416604893.
[I 2024-02-22 18:29:14,391] Trial 2 finished with value: 0.9285288862418106 and parameters: {'learning_rate': 0.0223865137998212, 'depth': 9, 'subsample': 0.3652183675598023, 'colsample_bylevel': 0.4246233761717247, 'min_data_in_leaf': 36}. Best is trial 2 with value: 0.9285288862418106.
[I 2024-02-22 18:29:26,547] Trial 3 finished with value: 0.9256027214909037 and parameters: {'learning_rate': 0.00413796347723

Number of completed trials: 50
Best trial:
	Best Score: 0.9286671630677588
	Best Params: 
    learning_rate: 0.04201325418144026
    depth: 7
    subsample: 0.44890863827770777
    colsample_bylevel: 0.6023507085950695
    min_data_in_leaf: 50


In [122]:
# visualize parameter importance
optuna.visualization.plot_param_importances(study)

In [123]:
# visualize study optimization
optuna.visualization.plot_optimization_history(study)

In [126]:
# final model
final_model = CatBoostClassifier(**trial.params)
final_model.fit(
    X_train,
    y_train,
    eval_set=[(X_test, y_test)],
    cat_features=cat_features,
    verbose=0,
    early_stopping_rounds=100,
)

y_pred = final_model.predict(X_test)
pred_labels = np.rint(y_pred)
score = f1_score(y_test, pred_labels)
print(f"the f1-score of the final model is {score}")

the f1-score of the final model is 0.9286671630677588


In [22]:
# saving model
final_model.save_model(fname="../models/model.json", format="json")